In [ ]:
from matplotlib import pyplot as pl

%matplotlib inline
import mpld3
import numpy as np

mpld3.enable_notebook()

# Image normalization and corrections when doing FFTs between the uv and lm domains

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/processing_functions_tutorials/imaging/demo_fft_ifft_handling.ipynb)

## Description

When we move gridded visibilities between the *uv* (aperture) domain and the *lm* (sky) domain we have to Fourier transform. To recover correct pixel values, and to reduce numerical artefacts, a few practical steps are needed that are usually glossed over in theory:

- **FFT normalization.** A discrete inverse FFT carries an implicit factor of `1/N` (`N` = number of samples transformed). When we image *gridded* visibilities we apply only the inverse transform, so this factor has to be normalized out (multiply by `n_l * n_m`) and the image divided by the sum of the visibility weights.
- **Gridding-convolution (prolate-spheroidal) correction.** Gridding convolves the visibilities with a kernel; in the image domain this multiplies the sky by the Fourier transform of that kernel (a smooth taper, `corrTerm`, that dims the field toward its edges). To get correct flux densities we divide the dirty image by `corrTerm`.
- **Padding.** We often pad an image (equivalently, oversample the uv grid) before the FFT to reduce aliasing and Gibbs ringing, and so that the spheroidal-correction blow-up at the very edge (where `corrTerm -> 0`) lands in a margin that is cropped away. The padding is added before the transform and removed before the final image is presented.

## Assumptions

The centre pixel is taken to be the phase centre.

## Install AstroVIPER

Skip this cell if you don't want to install the latest version of AstroVIPER.

In [ ]:
import os
from importlib.metadata import version

try:
    os.system("pip install --upgrade astroviper")

    import astroviper  # noqa: F401 -- availability probe for the pip-install fallback

    print("Using astroviper version", version("astroviper"))

except ImportError as exc:
    print(f"Could not import astroviper: {exc}")

## API note

This tutorial has been updated to the current processing-function API. The old `astroviper.core.imaging.imaging_utils.standard_image_grid_preparation` helpers used in earlier versions (`make_image_xds`, `make_empty_padded_uv_image`, `grid2xradio_spheroid_ms4`, `ifft_to_lm` / `fft_to_uv`, `correct_ifft_to_lm` / `correct_fft_to_uv`, `remove_padding`, `apply_pb`) have been consolidated. The padding, FFT normalization and prolate-spheroidal correction are now handled by the FFT/normalization gridder:

- **Low-level transforms:** `fft_lm_to_uv`, `ifft_uv_to_lm` (plus `add_padding` / `remove_padding`) from `astroviper.processing_functions.imaging.fft_normalize_prolate_spheriodal_gridder`.
- **Gridding-correction term:** `create_prolate_spheroidal_kernel` from `astroviper.processing_functions.imaging.gridding_convolution_functions.gcf_prolate_spheroidal`.
- **High-level wrappers on an image dataset:** `ifft_norm_img_xds` / `fft_norm_img_xds` (same module) encapsulate padding + FFT + normalization + spheroidal correction in one call. Image datasets are now built with `make_empty_sky_image` from `xradio.image` (which replaces the removed `make_image_xds`).

Below we demonstrate the same concepts with these current functions.

In [ ]:
from astroviper.processing_functions.imaging.fft_normalize_prolate_spheriodal_gridder import (
    add_padding,
    fft_lm_to_uv,
    ifft_norm_img_xds,
    ifft_uv_to_lm,
    remove_padding,
)
from astroviper.processing_functions.imaging.gridding_convolution_functions.gcf_prolate_spheroidal import (
    create_prolate_spheroidal_kernel,
)

# Use the always-available scipy FFT backend for reproducibility in this tutorial.
# (pyfftw is the default in the pipeline and is a drop-in replacement.)
FFT_BACKEND = "scipy"

## 1. The low-level FFT pair: sky (lm) &harr; uv grid

`fft_lm_to_uv` and `ifft_uv_to_lm` implement the standard radio-astronomy convention: `ifftshift` the input to move the centre to the array origin, transform, then `fftshift` the result back to a centred array. `fft_lm_to_uv` returns the complex uv grid; `ifft_uv_to_lm` returns the complex sky (take `.real` for a real sky).

We start with a synthetic sky containing a few point sources.

In [ ]:
npix = 256
sky = np.zeros((npix, npix))
# (row, col, flux) point sources, some placed away from the centre
sources = [(128, 128, 1.0), (90, 168, 0.7), (168, 96, 0.5), (64, 64, 0.3)]
for row, col, flux in sources:
    sky[row, col] = flux

pl.figure()
pl.imshow(sky, origin="lower")
pl.colorbar()
pl.title("synthetic sky (lm domain)")

Forward transform the sky into the uv grid. The result is complex; we plot its magnitude.

In [ ]:
uv = fft_lm_to_uv(sky, fft_backend=FFT_BACKEND)
print("uv grid dtype:", uv.dtype, " shape:", uv.shape)

pl.figure()
pl.imshow(np.abs(uv), origin="lower")
pl.colorbar()
pl.title("|uv grid| = |FFT(sky)|")

Inverse transform back to the sky. For a *pure* round trip the `1/N` normalization of the inverse transform is exactly undone by the forward transform, so the sky is recovered to machine precision with no rescaling. (When imaging *gridded* visibilities you apply only the inverse transform, so you must multiply by `npix*npix` and divide by the sum of weights to restore the flux scale &mdash; see `demo_standard_grid.ipynb`.)

In [ ]:
sky_roundtrip = ifft_uv_to_lm(uv, fft_backend=FFT_BACKEND).real
print("max round-trip error:", np.max(np.abs(sky_roundtrip - sky)))

pl.figure()
pl.imshow(sky_roundtrip, origin="lower")
pl.colorbar()
pl.title("round trip: ifft_uv_to_lm(fft_lm_to_uv(sky))")

## 2. Prolate-spheroidal gridding correction

During gridding the visibilities are convolved with a prolate-spheroidal kernel. In the image domain this multiplies the sky by the Fourier transform of the kernel, `corrTerm` &mdash; a smooth bump that is ~1 at the centre and tapers to ~0 at the field edge. `create_prolate_spheroidal_kernel(oversampling, support, image_size)` returns the gridding `kernel` and this image-domain `corrTerm`. To recover correct flux densities we **divide** the dirty image by `corrTerm`.

In [ ]:
oversampling = 100
support = 7
kernel, corrTerm = create_prolate_spheroidal_kernel(
    oversampling, support, np.array([npix, npix])
)
print("corrTerm  centre:", corrTerm[npix // 2, npix // 2], " edge:", corrTerm[0, 0])

pl.figure()
pl.imshow(corrTerm, origin="lower")
pl.colorbar()
pl.title("corrTerm: image-domain gridding taper")

To see the effect, take an extended sky (a few Gaussian blobs, some near the edges), apply the taper `corrTerm` (this is what an *uncorrected* dirty image looks like), then divide by `corrTerm` to remove it. Because the taper multiplies the sky in the image domain, dividing it out recovers the original brightness exactly.

In [ ]:
rows, cols = np.mgrid[0:npix, 0:npix]


def blob(r, c, amp, sigma=7.0):
    return amp * np.exp(-(((rows - r) ** 2 + (cols - c) ** 2) / (2.0 * sigma**2)))


sky_ext = (
    blob(128, 128, 1.0) + blob(40, 40, 1.0) + blob(215, 60, 1.0) + blob(60, 210, 1.0)
)

tapered = sky_ext * corrTerm  # what the gridding convolution function imprints

pl.figure()
pl.imshow(tapered, origin="lower")
pl.colorbar()
pl.title("uncorrected dirty image (tapered by corrTerm)")

In [ ]:
corrected = tapered / corrTerm  # spheroidal correction
print("max correction error:", np.max(np.abs(corrected - sky_ext)))

pl.figure()
pl.imshow(corrected, origin="lower")
pl.colorbar()
pl.title("after dividing by corrTerm (taper removed)")

## 3. Padding before the FFT

Padding the image (oversampling the uv grid) reduces aliasing and Gibbs ringing, and it keeps the spheroidal-correction blow-up at the very edge &mdash; where `corrTerm -> 0` &mdash; inside a margin that is cropped away. `add_padding` embeds a small image into the centre of a larger zero-filled buffer; `remove_padding` crops the central region back out. Here we pad the sky, round-trip it through the uv domain, then crop back to the original size.

In [ ]:
npad = 320
padded = np.zeros((npad, npad))
add_padding(sky, padded)  # embed the 256x256 sky at the centre of a 320x320 buffer

uv_padded = fft_lm_to_uv(padded, fft_backend=FFT_BACKEND)
sky_padded_rt = ifft_uv_to_lm(uv_padded, fft_backend=FFT_BACKEND).real
cropped = remove_padding(sky_padded_rt, [npix, npix])  # crop back to 256x256

print("padded shape:", padded.shape, " cropped shape:", cropped.shape)
print("padded round-trip + crop error:", np.max(np.abs(cropped - sky)))

pl.figure()
pl.imshow(cropped, origin="lower")
pl.colorbar()
pl.title("padded round trip, then remove_padding")

## 4. High-level wrappers on an image dataset

In the imaging pipeline these operations act on an *xradio image dataset* (`img_xds`) rather than on bare numpy arrays. `ifft_norm_img_xds` performs, in a single call and plane by plane: the inverse FFT (uv &rarr; lm), the prolate-spheroidal correction, the flux/weight normalization, and the removal of padding &mdash; exactly the sequence the removed `make_empty_padded_uv_image` / `grid2xradio_spheroid_ms4` / `correct_ifft_to_lm` / `remove_padding` helpers used to do by hand. Its forward partner `fft_norm_img_xds` goes the other way (model sky &rarr; padded model uv grid).

Image datasets are built with `make_empty_sky_image` (from `xradio.image`), which replaces the old `make_image_xds`. Below we build a small image dataset, drop in a uniformly-weighted uv-sampling grid, and let `ifft_norm_img_xds` turn it into a point-spread function &mdash; the inverse transform of a flat sampling function is a PSF peaked at the centre. See `make_psf_demo.ipynb` for the full pipeline that grids real visibilities into this grid.

In [ ]:
import xarray as xr
from xradio.image import make_empty_sky_image

from astroviper.processing_functions.imaging.utils.fft_sizing import (
    next_fft_friendly_size,
)

image_size = [64, 64]
img_xds = make_empty_sky_image(
    phase_center=[0.0, 0.0],
    image_size=image_size,
    cell_size=[-1.0e-5, 1.0e-5],
    frequency_coords=[1.0e9],
    pol_coords=["I"],
    time_coords=[0],
)
img_xds.attrs["type"] = "image_dataset"
# The FFT/normalization wrappers are driven by data groups; add one to write into.
img_xds = img_xds.xr_img.add_data_group(
    new_data_group_name="residual",
    new_data_group={"description": "fft/ifft handling demo", "date": "2026"},
)
img_xds

In [ ]:
# A uniformly-sampled (flat) uv grid; its inverse transform is the PSF.
# The grid is padded relative to the image (padding removed inside the wrapper).
n_uv = [next_fft_friendly_size(int(np.ceil(s * 1.2)), even=True) for s in image_size]
print("image_size:", image_size, " padded uv size:", n_uv)

uv_grid = np.ones((1, 1, 1, n_uv[0], n_uv[1]), dtype=np.complex128)
sum_weight = np.full((1, 1, 1), float(np.abs(uv_grid[0, 0, 0]).sum()))  # sum of weights

img_xds["UV_SAMPLING"] = xr.DataArray(
    uv_grid, dims=("time", "frequency", "polarization", "u", "v")
)
img_xds["UV_SAMPLING_NORMALIZATION"] = xr.DataArray(
    sum_weight, dims=("time", "frequency", "polarization")
)
# Register the grid and its normalization in the input data group.
data_group = img_xds.attrs["data_groups"]["residual"]
data_group["uv_sampling"] = "UV_SAMPLING"
data_group["uv_sampling_normalization"] = "UV_SAMPLING_NORMALIZATION"

In [ ]:
img_xds = ifft_norm_img_xds(
    img_xds,
    image_params={"image_size": image_size},
    image_data_group_in_name="residual",
    image_data_group_out_name="residual",
    image_data_group_out_modified={"point_spread_function": "POINT_SPREAD_FUNCTION"},
    image_data_variables_keep=["uv_sampling"],
    processing_function_threads=1,
    fft_backend=FFT_BACKEND,
)

psf = img_xds["POINT_SPREAD_FUNCTION"].values[0, 0, 0]
print(
    "PSF shape:",
    psf.shape,
    " peak:",
    psf.max(),
    " at pixel:",
    np.unravel_index(np.argmax(psf), psf.shape),
)

pl.figure()
pl.imshow(psf, origin="lower")
pl.colorbar()
pl.title("PSF from ifft_norm_img_xds (padding + norm + spheroidal corr. internal)")